In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import wandb
import os
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances_argmin_min


In [3]:
# Constants
NC = 24
NIC = 1
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# --- Multi-GPU Device Setup ---
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for Data Parallelism.")
    device = torch.device("cuda:0")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using single device: {device}")

# --- Output Directory Structure ---
OUTPUT_BASE = './output/nic1'
MODEL_CHECKPOINT_DIR = os.path.join(OUTPUT_BASE, 'model_checkpoints')
MODEL_LOGS_DIR = os.path.join(OUTPUT_BASE, 'models')
OUTPUT_IMAGES_DIR = os.path.join(OUTPUT_BASE, 'output_images')

# Create all directories
os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_LOGS_DIR, exist_ok=True)
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)

CENTROID_PATH = './data/color_centroids.npy'

Using single device: cuda


In [4]:
# Add this at the beginning to debug CUDA issues
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
else:
    print("CUDA is not available. Running on CPU.")

PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA version: 12.8
Number of GPUs: 1
GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU
  Memory: 8.2 GB


In [5]:
# ============================================================================
# DATASET
# ============================================================================
class ColorizationDataset(Dataset):
    def __init__(self, cifar_dataset, centroids, is_train=True):
        self.cifar_dataset = cifar_dataset
        self.centroids = centroids.astype(np.float32)
        self.is_train = is_train
        self.to_tensor = transforms.ToTensor()

        if is_train:
            self.augment = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
            ])
        else:
            self.augment = None

    def __len__(self):
        return len(self.cifar_dataset)

    def __getitem__(self, idx):
            img_pil, _ = self.cifar_dataset[idx]

            # Augmentation
            if self.augment is not None:
                img_pil = self.augment(img_pil)

            # Convert to grayscale for input (NIC=1)
            img_gray = transforms.Grayscale(num_output_channels=1)(img_pil)
            img_gray_input = self.to_tensor(img_gray)  # [1, 32, 32]

            # RGB for label computation (use original color image)
            img_rgb = self.to_tensor(img_pil)  # [3, 32, 32]
            img_rgb_label_comp = img_rgb.permute(1, 2, 0).reshape(-1, 3).numpy()
            img_rgb_label_comp = img_rgb_label_comp.astype(np.float32)

            # Get nearest centroid labels
            labels = pairwise_distances_argmin_min(img_rgb_label_comp, self.centroids, metric='euclidean')[0]
            label_map = torch.tensor(labels, dtype=torch.long).reshape(32, 32)

            return img_gray_input, label_map 

def load_cifar_data(batch_size=64, centroids=None):
    """Load CIFAR-10 data."""
    # Load datasets
    full_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
    test_raw = datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

    # Split train/val (90/10)
    train_size = int(0.9 * len(full_train))
    val_size = len(full_train) - train_size
    train_subset, val_subset = random_split(full_train, [train_size, val_size])

    # Create wrapper datasets
    train_dataset = ColorizationDataset(
        [full_train[i] for i in train_subset.indices],
        centroids,
        is_train=True
    )

    val_dataset = ColorizationDataset(
        [full_train[i] for i in val_subset.indices],
        centroids,
        is_train=False
    )

    test_dataset = ColorizationDataset(test_raw, centroids, is_train=False)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, test_dataset



In [6]:
# ============================================================================
# MODEL
# ============================================================================
class ColorizationCNN(nn.Module):
    def __init__(self, NIC=1, NF=32, NC=24, kernel_size=3, dropout=0.3):
        super(ColorizationCNN, self).__init__()

        padding = kernel_size // 2

        # Encoder
        self.enc1 = self._conv_block(NIC, NF, kernel_size, padding)
        self.enc2 = self._conv_block(NF, 2 * NF, kernel_size, padding)
        self.enc3 = self._conv_block(2 * NF, 4 * NF, kernel_size, padding)

        # Decoder
        self.dec1 = self._deconv_block(4 * NF, 2 * NF, dropout=dropout)
        self.dec2 = self._deconv_block(2 * NF, NF, dropout=dropout)
        self.dec3 = self._deconv_block(NF, NC, dropout=0.0, final=True)

        # Classifier
        self.classifier = nn.Conv2d(NC, NC, kernel_size=1)

        self._initialize_weights()

    def _conv_block(self, in_c, out_c, kernel_size, padding):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

    def _deconv_block(self, in_c, out_c, dropout=0.3, final=False):
        layers = [
            nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_c)
        ]
        if not final:
            layers.append(nn.ReLU(inplace=True))
            if dropout > 0:
                layers.append(nn.Dropout2d(dropout))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)
        x = self.classifier(x)
        return x



In [7]:
# ============================================================================
# TRAINING
# ============================================================================
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0

    for batch_idx, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += pixel_accuracy(outputs, targets)

        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)}: Loss={loss.item():.4f}')

    return total_loss / len(loader), total_acc / len(loader)


def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            total_acc += pixel_accuracy(outputs, targets)

    return total_loss / len(loader), total_acc / len(loader)


@torch.no_grad()
def pixel_accuracy(logits, targets):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == targets).sum().item()
    total = preds.numel()
    return correct / total


def compute_class_weights(train_loader, NC, device):
    """Compute balanced class weights from actual data."""
    print("\nComputing class weights from training data...")
    all_labels = []

    for batch_idx, (_, labels) in enumerate(train_loader):
        all_labels.append(labels.view(-1).cpu().numpy())
        if batch_idx >= 100:  # Sample first 100 batches for speed
            break

    all_labels = np.concatenate(all_labels)
    counts = np.bincount(all_labels, minlength=NC)

    print(f"Class distribution (first 100 batches):")
    for i in range(NC):
        if counts[i] > 0:
            pct = 100 * counts[i] / counts.sum()
            print(f"  Cluster {i:2d}: {counts[i]:7d} pixels ({pct:5.2f}%)")

    # Compute weights
    freq = counts / counts.sum()
    weights = 1.0 / (freq + 0.01)  # Smoothing
    weights = weights / weights.sum() * NC

    return torch.tensor(weights, dtype=torch.float32).to(device)


def logits_to_rgb(logits, centroids):
    """Convert logits to RGB using centroids."""
    device = logits.device
    B, NC, H, W = logits.shape
    preds = torch.argmax(logits, dim=1).view(-1)
    centroids_t = torch.from_numpy(centroids).to(device)
    rgb_flat = centroids_t[preds]
    return rgb_flat.view(B, H, W, 3).permute(0, 3, 1, 2)


def generate_colorized_images(model, test_dataset, centroids, device, num_images=10):
    """Generate and display colorized images."""
    model.eval()
    fig, axes = plt.subplots(3, num_images, figsize=(num_images * 2, 6))

    with torch.no_grad():
        for i in range(num_images):
            # The input is now RGB, not grayscale
            rgb_input, _ = test_dataset[i]
            original_rgb = transforms.ToTensor()(test_dataset.cifar_dataset[i][0])

            rgb_batch = rgb_input.unsqueeze(0).to(device)
            logits = model(rgb_batch)
            predicted_rgb = logits_to_rgb(logits.cpu(), centroids).squeeze(0)

            # Plot
            axes[0, i].imshow(original_rgb.permute(1, 2, 0))
            axes[0, i].axis('off')
            axes[1, i].imshow(rgb_input.permute(1, 2, 0)) # Show RGB input
            axes[1, i].axis('off')
            axes[2, i].imshow(predicted_rgb.permute(1, 2, 0).clamp(0, 1))
            axes[2, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=12)
    axes[1, 0].set_ylabel('Grayscale Input', fontsize=12) # <-- MODIFIED Label
    axes[2, 0].set_ylabel('Predicted Cluster', fontsize=12) # <-- MODIFIED Label

    # Save to output_images directory
    output_path = os.path.join(OUTPUT_IMAGES_DIR, 'colorized_results.png')
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"Saved colorized results to {output_path}")

    try:
        wandb.log({"colorized_images": wandb.Image(plt)})
    except:
        pass

    plt.show()

In [8]:
def main():
    # Load centroids
    centroids = np.load(CENTROID_PATH)
    if centroids.max() > 1.5:
        centroids = centroids / 255.0
    centroids = centroids.astype(np.float32)
    print(f"Loaded centroids: {centroids.shape}, range [{centroids.min():.4f}, {centroids.max():.4f}]")

    # Load data
    print("\nLoading CIFAR-10 data...")
    train_loader, val_loader, test_loader, test_dataset = load_cifar_data(
        batch_size=64,
        centroids=centroids
    )

    # Model
    model = ColorizationCNN(NIC=1, NF=32, NC=24, kernel_size=3, dropout=0.3)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        model.to(device)
    else:
        model.to(device)

    print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Loss and optimizer
    class_weights = compute_class_weights(train_loader, NC, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    if isinstance(model, nn.DataParallel):
        params = model.module.parameters()
    else:
        params = model.parameters()

    optimizer = torch.optim.AdamW(params, lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    # WandB
    wandb.init(project="cifar10-colorization-sweep", config={
        "learning_rate": 3e-4,
        "epochs": 25,
        "batch_size": 64,
        "NF": 32,
        "dropout": 0.3,
        "optimizer": "AdamW",
        "label_smoothing": 0.1,
        "NIC": 1
    })

    # Training loop
    best_val_loss = float("inf")
    patience_counter = 0
    PATIENCE = 7
    EPOCHS = 25

    for epoch in range(EPOCHS):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"{'='*60}")

        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "lr": optimizer.param_groups[0]['lr']
        })

        # Save best model to model_checkpoints directory
        if val_loss < best_val_loss - 1e-4:
            print(f"✓ Validation improved: {best_val_loss:.4f} → {val_loss:.4f}")
            best_val_loss = val_loss
            patience_counter = 0
            state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            checkpoint_path = os.path.join(MODEL_CHECKPOINT_DIR, "best_colorization_model.pt")
            torch.save({
                'epoch': epoch,
                'model_state_dict': state_dict_to_save,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, checkpoint_path)
            print(f"Saved checkpoint to {checkpoint_path}")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")

            if patience_counter >= PATIENCE:
                print("Early stopping triggered.")
                break

    # Load best and visualize
    checkpoint_path = os.path.join(MODEL_CHECKPOINT_DIR, "best_colorization_model.pt")
    checkpoint = torch.load(checkpoint_path)
    final_model = ColorizationCNN(NIC=NIC, NF=32, NC=24, kernel_size=3, dropout=0.3).to(device)
    final_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\n✓ Loaded best model from epoch {checkpoint['epoch']+1}")

    generate_colorized_images(final_model, test_dataset, centroids, device, num_images=10)

    wandb.finish()
    print("\n✓ Training complete!")

In [9]:
def generate_colorization_figure(model, test_dataset, centroids, device, num_images=8):
    """
    Generate a figure showing original, grayscale input, and predicted cluster RGB images.
    """
    model.eval()

    fig, axes = plt.subplots(3, num_images, figsize=(num_images * 2, 6))

    with torch.no_grad():
        for i in range(num_images):
            # Get grayscale input and original RGB
            gray_input, _ = test_dataset[i]
            original_rgb = transforms.ToTensor()(test_dataset.cifar_dataset[i][0])

            # Generate colorized prediction
            gray_batch = gray_input.unsqueeze(0).to(device)  # [1, 1, 32, 32]
            logits = model(gray_batch)  # [1, 24, 32, 32]

            # Convert logits to RGB
            predicted_rgb = logits_to_rgb(logits, centroids)
            predicted_rgb = predicted_rgb.squeeze(0).cpu()  # [3, 32, 32]

            # Plot
            axes[0, i].imshow(original_rgb.permute(1, 2, 0))
            axes[0, i].axis('off')

            axes[1, i].imshow(gray_input.squeeze(0), cmap='gray')  # Show grayscale
            axes[1, i].axis('off')

            axes[2, i].imshow(predicted_rgb.permute(1, 2, 0).clamp(0, 1))
            axes[2, i].axis('off')

    # Add row labels
    axes[0, 0].set_ylabel('Original', fontsize=12, rotation=0, labelpad=40, va='center')
    axes[1, 0].set_ylabel('Grayscale Input', fontsize=12, rotation=0, labelpad=40, va='center')
    axes[2, 0].set_ylabel('Predicted Cluster', fontsize=12, rotation=0, labelpad=40, va='center')

    plt.tight_layout()

    return fig

In [ ]:
# ============================================================================
# STEP 1: Define Sweep Configuration
# ============================================================================
sweep_config = {
    "name": "cifar10-cluster-prediction-hyperparameter-sweep", # <-- MODIFIED name
    "method": "random",
    "metric": {
        "name": "best_val_loss",
        "goal": "minimize"
    },
    "parameters": {
        "learning_rate": {
            "values": [1e-4, 3e-4, 1e-3, 3e-3]
        },
        "batch_size": {
            "values": [32, 64, 128]
        },
        "num_filters": {
            "values": [8, 16, 32]
        },
        "kernel_size": {
            "values": [3, 5]
        },
        "optimizer": {
            "values": ["SGD", "Adam"]
        },
        "dropout": {
            "values": [0.2, 0.3, 0.4]
        }
    }
}

# ============================================================================
# STEP 2: Define Training Function for Sweep
# ============================================================================
def train_sweep_run(config=None):
    """
    Training function called by wandb agent for each sweep configuration.
    """
    with wandb.init(config=config):
        # ========== EXPLICIT DEVICE SETUP ==========
        if torch.cuda.is_available():
            current_device = torch.device("cuda")
            print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
            torch.cuda.empty_cache()
        else:
            current_device = torch.device("cpu")
            print("⚠ WARNING: Using CPU - training will be very slow!")
        # ===========================================
        
        cfg = wandb.config

        # Load centroids
        centroids = np.load(CENTROID_PATH)
        if centroids.max() > 1.5:
            centroids = centroids / 255.0
        centroids = centroids.astype(np.float32)

        # Load data with sweep batch size
        train_loader, val_loader, test_loader, test_dataset = load_cifar_data(
            batch_size=cfg.batch_size,
            centroids=centroids
        )

        # Initialize model with sweep parameters (NIC=1 for grayscale)
        model = ColorizationCNN(
            NIC=1,  # Fixed to 1 for grayscale input
            NF=cfg.num_filters,
            NC=24,
            kernel_size=cfg.kernel_size,
            dropout=cfg.dropout
        )

        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model) 
            model.to(current_device)
        else:
            model.to(current_device)

        # Compute class weights
        class_weights = compute_class_weights(train_loader, NC, current_device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        
        if isinstance(model, nn.DataParallel):
            params = model.module.parameters()
        else:
            params = model.parameters()

        # Optimizer based on config
        if cfg.optimizer == "SGD":
            optimizer = torch.optim.SGD(
                params,
                lr=cfg.learning_rate,
                momentum=0.9,
                weight_decay=1e-4
            )
        else:  # Adam
            optimizer = torch.optim.AdamW(
                params,
                lr=cfg.learning_rate,
                weight_decay=1e-4
            )

        # Scheduler
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )

        # Training loop
        best_val_loss = float("inf")
        patience_counter = 0
        PATIENCE = 3
        EPOCHS = 25

        for epoch in range(EPOCHS):
            # Train
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, current_device)

            # Validate
            val_loss, val_acc = validate_epoch(model, val_loader, criterion, current_device)

            # Update scheduler
            scheduler.step(val_loss)

            # Log to wandb
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
                "learning_rate": optimizer.param_groups[0]['lr']
            })

            print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

            # Track best model - save to model_checkpoints directory
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                state_dict_to_save = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
                
                checkpoint_path = os.path.join(MODEL_CHECKPOINT_DIR, f"best_{wandb.run.id}.pt")
                torch.save({
                    'model_state_dict': state_dict_to_save,
                    'val_loss': val_loss,
                    'config': dict(cfg)
                }, checkpoint_path)
                print(f"Saved checkpoint to {checkpoint_path}")

            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"Early stopping at epoch {epoch+1}")
                    break

        # Log best validation loss
        wandb.log({"best_val_loss": best_val_loss})

        # Generate example images with best model
        checkpoint_path = os.path.join(MODEL_CHECKPOINT_DIR, f"best_{wandb.run.id}.pt")
        checkpoint = torch.load(checkpoint_path)
        
        final_model = ColorizationCNN(
            NIC=1,
            NF=cfg.num_filters,
            NC=24,
            kernel_size=cfg.kernel_size,
            dropout=cfg.dropout
        ).to(current_device)
        
        final_model.load_state_dict(checkpoint['model_state_dict']) 

        # Generate example colorizations with best model
        print("\nGenerating example images...")
        fig = generate_colorization_figure(final_model, test_dataset, centroids, current_device, num_images=8)

        # Save figure to output_images directory
        fig_path = os.path.join(OUTPUT_IMAGES_DIR, f"sweep_{wandb.run.id}.png")
        fig.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f"Saved example images to {fig_path}")

        # Log final results
        wandb.log({
            "best_val_loss": best_val_loss,
            "example_colorizations": wandb.Image(fig)
        })
        plt.close(fig)

        # Clean up memory
        del model, optimizer, train_loader, val_loader, test_loader
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return best_val_loss

# ============================================================================
# STEP 3: Initialize and Run Sweep
# ============================================================================

# Initialize the sweep
sweep_id = wandb.sweep(
    sweep_config,
    project="cifar10-colorization-sweep" # <-- MODIFIED project name
)

print(f"Sweep ID: {sweep_id}")
print(f"Sweep URL will be available after first run starts")

# Run the sweep agent (20 configurations)
wandb.agent(sweep_id, function=train_sweep_run, count=20)

print(f"\n✓ Sweep completed!")
print(f"View results at: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep/sweeps/{sweep_id}")



In [ ]:
# %pip install pandas
# import pandas as pd

In [15]:
def print_best_results_robust(current_sweep_id):
    """
    Analyzes sweep results by calculating the minimum val_loss from run history,
    bypassing corrupted or missing summary data. This function is robust to the
    case where pandas is not available (wandb will return a list instead of a DataFrame).
    """
    api = wandb.Api()

    try:
        sweep = api.sweep(f"gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep/{current_sweep_id}")
    except Exception as e:
        print(f"\nError retrieving sweep. Check username and project name. Details: {e}")
        return None

    run_metrics = []
    print("\nAttempting robust data retrieval from run history...")

    # --- 1. Retrieve and Calculate Best Loss from History for Each Run ---
    for run in sweep.runs:
        try:
            # Request history without forcing pandas (works when pandas is missing)
            history = run.history(keys=['val_loss'], pandas=False)

            # history may be a list (no pandas) or a pandas.DataFrame (if pandas present)
            min_loss = float("inf")
            try:
                # If history is a list of dicts
                if isinstance(history, list):
                    vals = [h.get('val_loss') for h in history if isinstance(h, dict) and 'val_loss' in h and h.get('val_loss') is not None]
                    if len(vals) > 0:
                        min_loss = float(min(vals))
                    else:
                        print(f"Warning: Run {run.name} ({run.id}) has no 'val_loss' entries in history.")
                else:
                    # Assume a pandas-like object
                    if hasattr(history, 'empty') and history.empty:
                        print(f"Warning: Run {run.name} ({run.id}) has an empty history DataFrame.")
                    elif hasattr(history, 'columns') and 'val_loss' not in history.columns:
                        print(f"Warning: Run {run.name} ({run.id}) history lacks 'val_loss' column.")
                    else:
                        # pandas DataFrame case
                        min_loss = float(history['val_loss'].min())
            except Exception as inner_e:
                print(f"Warning: could not parse history for run {run.name} ({run.id}): {inner_e}")
                min_loss = float("inf")

            # Safely get config - convert to dict if it's not already
            config = run.config
            if isinstance(config, str):
                import json
                try:
                    config = json.loads(config)
                except:
                    config = {}
            elif not isinstance(config, dict):
                config = dict(config) if hasattr(config, '__iter__') else {}

            run_metrics.append({
                'run': run,
                'min_val_loss': min_loss,
                'config': config
            })

        except Exception as e:
            # Catch errors during history fetching
            print(f"ERROR: Failed to retrieve history for run {run.name} ({run.id}). Skipping. Error: {e}")
            run_metrics.append({
                'run': run,
                'min_val_loss': float("inf"),
                'config': {}
            })

    # --- 2. Sort Runs by the Calculated Minimum Loss ---
    runs_sorted = sorted(run_metrics, key=lambda x: x['min_val_loss'])

    if not runs_sorted or runs_sorted[0]['min_val_loss'] == float("inf"):
        print("\n" + "="*70)
        print("COULD NOT DETERMINE BEST RUN: All runs failed or lack history data.")
        print("="*70)
        return None

    # --- 3. Print Results ---
    best_result = runs_sorted[0]
    best_run = best_result['run']
    
    print("\n" + "="*70)
    print("BEST HYPERPARAMETER CONFIGURATION (NIC=3) - HISTORY ANALYSIS")
    print("="*70)

    print(f"\nRun Name: {best_run.name}")
    print(f"Run ID: {best_run.id}")
    print(f"Run URL: {best_run.url}")
    print(f"\nCalculated Best Validation Loss: {best_result['min_val_loss']:.6f}")

    print(f"\nOptimal Hyperparameters:")
    config = best_result['config']
    if config:
        print(f"  Learning Rate:  {config.get('learning_rate', 'N/A')}")
        print(f"  Batch Size:     {config.get('batch_size', 'N/A')}")
        print(f"  Num Filters:    {config.get('num_filters', 'N/A')}")
        print(f"  Kernel Size:    {config.get('kernel_size', 'N/A')}")
        print(f"  Optimizer:      {config.get('optimizer', 'N/A')}")
        print(f"  Dropout:        {config.get('dropout', 'N/A')}")
    else:
        print("  No config data available")

    print(f"\n" + "="*70)
    print("TOP 5 CONFIGURATIONS")
    print("="*70)

    for i, result in enumerate(runs_sorted[:5], 1):
        loss_str = f"{result['min_val_loss']:.6f}" if result['min_val_loss'] != float("inf") else "FAILED/N/A"
        config = result['config']
        
        print(f"\n{i}. {result['run'].name}")
        print(f"   Val Loss: {loss_str}")
        if config:
            print(f"   LR={config.get('learning_rate', 'N/A')}, "
                  f"BS={config.get('batch_size', 'N/A')}, "
                  f"NF={config.get('num_filters', 'N/A')}, "
                  f"KS={config.get('kernel_size', 'N/A')}")
            print(f"   Optimizer={config.get('optimizer', 'N/A')}, "
                  f"Dropout={config.get('dropout', 'N/A')}")
        else:
            print("   No config available")
        print(f"   URL: {result['run'].url}")

    print("\n" + "="*70)

    return best_run

# Run the analysis
best_run = print_best_results_robust(sweep_id)


Attempting robust data retrieval from run history...

BEST HYPERPARAMETER CONFIGURATION (NIC=3) - HISTORY ANALYSIS

Run Name: glorious-sweep-2
Run ID: 2g49e3j0
Run URL: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep/runs/2g49e3j0

Calculated Best Validation Loss: 2.204352

Optimal Hyperparameters:
  Learning Rate:  0.003
  Batch Size:     128
  Num Filters:    32
  Kernel Size:    5
  Optimizer:      Adam
  Dropout:        0.2

TOP 5 CONFIGURATIONS

1. glorious-sweep-2
   Val Loss: 2.204352
   LR=0.003, BS=128, NF=32, KS=5
   Optimizer=Adam, Dropout=0.2
   URL: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep/runs/2g49e3j0

2. dandy-sweep-12
   Val Loss: 2.297776
   LR=0.001, BS=32, NF=32, KS=3
   Optimizer=Adam, Dropout=0.3
   URL: https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep/runs/wn9pyqay

3. amber-sweep-19
   Val Loss: 2.412491
   LR=0.001, BS=128, NF=16, KS=3
   Optimizer=Adam, Dropout=0.3
   URL: https:

# Analytical Questions
## Model Architecture: Image Colorization CNN

---

## Architecture Specification and Assumptions

### Model Architecture:
```
Input: [B, NIC, H, W]

ENCODER (Downsampling):
├─ Block 1: Conv2d → BatchNorm → ReLU → MaxPool2d → [B, NF, H/2, H/2]
├─ Block 2: Conv2d → BatchNorm → ReLU → MaxPool2d → [B, 2NF, H/4, H/4]
└─ Block 3: Conv2d → BatchNorm → ReLU → MaxPool2d → [B, 4NF, H/8, H/8]

DECODER (Upsampling):
├─ Block 4: ConvTranspose2d → BatchNorm → ReLU → [B, 2NF, H/4, H/4]
├─ Block 5: ConvTranspose2d → BatchNorm → ReLU → [B, NF, H/2, H/2]
└─ Block 6: ConvTranspose2d → BatchNorm → ReLU → [B, NC, H, H]

CLASSIFIER:
└─ Conv2d (1×1) → [B, NC, H, H]
```

### Key Assumptions:

**Encoder Convolutions:**
- Kernel size: k × k (typically k=3 or k=5)
- Stride: 1
- Padding: p = ⌊k/2⌋ (maintains spatial dimensions)
  - For k=3: padding=1
  - For k=5: padding=2
- MaxPool: kernel=2, stride=2 (halves dimensions)

**Decoder Transpose Convolutions:**
- Kernel size: 2 × 2
- Stride: 2
- Padding: 0
- Output padding: 0
- Effect: doubles spatial dimensions

**Classifier:**
- Kernel size: 1 × 1
- Stride: 1
- Padding: 0
- Effect: maintains spatial dimensions, only changes channels

**Parameters to Ignore:**
- Biases (as stated)
- BatchNorm parameters (γ, β)

---

# Part 1: Analysis for H=W=32 (32×32 Input)

## Step 1: Number of Weights

### Formula for Convolutional Layer Weights:
For Conv2d(in_channels, out_channels, kernel_size):
```
Weights = in_channels × out_channels × kernel_height × kernel_width
```

### Encoder Layer 1: [NIC, 32, 32] → [NF, 32, 32]
- **Conv2d(NIC → NF, kernel=k, padding=⌊k/2⌋)**
  ```
  W₁ = NIC × NF × k × k = NIC·NF·k²
  ```
  - Input channels: NIC
  - Output channels: NF
  - Kernel: k×k
  - Example (k=3): W₁ = NIC·NF·9

- **MaxPool2d(kernel=2, stride=2)**: No parameters
- Output shape: [NF, 16, 16]

### Encoder Layer 2: [NF, 16, 16] → [2NF, 16, 16]
- **Conv2d(NF → 2NF, kernel=k, padding=⌊k/2⌋)**
  ```
  W₂ = NF × (2NF) × k × k = 2NF²·k²
  ```
  - Input channels: NF
  - Output channels: 2NF
  - Kernel: k×k
  - Example (k=3): W₂ = 2NF²·9 = 18NF²

- **MaxPool2d(kernel=2, stride=2)**: No parameters
- Output shape: [2NF, 8, 8]

### Encoder Layer 3: [2NF, 8, 8] → [4NF, 8, 8]
- **Conv2d(2NF → 4NF, kernel=k, padding=⌊k/2⌋)**
  ```
  W₃ = (2NF) × (4NF) × k × k = 8NF²·k²
  ```
  - Input channels: 2NF
  - Output channels: 4NF
  - Kernel: k×k
  - Example (k=3): W₃ = 8NF²·9 = 72NF²

- **MaxPool2d(kernel=2, stride=2)**: No parameters
- Output shape: [4NF, 4, 4]

### Decoder Layer 1: [4NF, 4, 4] → [2NF, 8, 8]
- **ConvTranspose2d(4NF → 2NF, kernel=2, stride=2)**
  ```
  W₄ = (4NF) × (2NF) × 2 × 2 = 16NF²
  ```
  - Input channels: 4NF
  - Output channels: 2NF
  - Kernel: 2×2 (fixed for transpose conv)
  - Output size: (4-1)×2 + 2 = 8

### Decoder Layer 2: [2NF, 8, 8] → [NF, 16, 16]
- **ConvTranspose2d(2NF → NF, kernel=2, stride=2)**
  ```
  W₅ = (2NF) × NF × 2 × 2 = 4NF²
  ```
  - Input channels: 2NF
  - Output channels: NF
  - Kernel: 2×2
  - Output size: (8-1)×2 + 2 = 16

### Decoder Layer 3: [NF, 16, 16] → [NC, 32, 32]
- **ConvTranspose2d(NF → NC, kernel=2, stride=2)**
  ```
  W₆ = NF × NC × 2 × 2 = 4NF·NC
  ```
  - Input channels: NF
  - Output channels: NC
  - Kernel: 2×2
  - Output size: (16-1)×2 + 2 = 32

### Classifier: [NC, 32, 32] → [NC, 32, 32]
- **Conv2d(NC → NC, kernel=1)**
  ```
  W₇ = NC × NC × 1 × 1 = NC²
  ```
  - Input channels: NC
  - Output channels: NC
  - Kernel: 1×1 (pointwise convolution)

### Total Weights (32×32):
```
W_total = W₁ + W₂ + W₃ + W₄ + W₅ + W₆ + W₇

W_total = NIC·NF·k² + 2NF²·k² + 8NF²·k² + 16NF² + 4NF² + 4NF·NC + NC²

W_total = NIC·NF·k² + (2 + 8)NF²·k² + (16 + 4)NF² + 4NF·NC + NC²

W_total = NIC·NF·k² + 10NF²·k² + 20NF² + 4NF·NC + NC²
```

**Factored form:**
```
W_total = NF·k²(NIC + 10NF) + 4NF(5NF + NC) + NC²
```

#### For k=3:
```
W_total = 9·NIC·NF + 90NF² + 20NF² + 4NF·NC + NC²
W_total = 9·NIC·NF + 110NF² + 4NF·NC + NC²
```

#### For k=5:
```
W_total = 25·NIC·NF + 250NF² + 20NF² + 4NF·NC + NC²
W_total = 25·NIC·NF + 270NF² + 4NF·NC + NC²
```

#### Example (NIC=1, NF=32, NC=24, k=3):
```
W_total = 9(1)(32) + 110(32²) + 4(32)(24) + 24²
        = 288 + 110(1024) + 3072 + 576
        = 288 + 112,640 + 3,072 + 576
        = 116,576 weights
```

---

## Step 2: Number of Outputs (Activations)

For each layer, we count the total number of activation elements for a **single input image** (batch size = 1).

### Spatial Dimension Tracking:

| Layer | Operation | Input Shape | Output Shape | # Elements |
|-------|-----------|-------------|--------------|------------|
| Input | - | [1, NIC, 32, 32] | - | - |
| **Encoder** |
| Conv1 | Conv+BN+ReLU | [1, NIC, 32, 32] | [1, NF, 32, 32] | NF × 32² = **1024·NF** |
| Pool1 | MaxPool | [1, NF, 32, 32] | [1, NF, 16, 16] | NF × 16² = **256·NF** |
| Conv2 | Conv+BN+ReLU | [1, NF, 16, 16] | [1, 2NF, 16, 16] | 2NF × 16² = **512·NF** |
| Pool2 | MaxPool | [1, 2NF, 16, 16] | [1, 2NF, 8, 8] | 2NF × 8² = **128·NF** |
| Conv3 | Conv+BN+ReLU | [1, 2NF, 8, 8] | [1, 4NF, 8, 8] | 4NF × 8² = **256·NF** |
| Pool3 | MaxPool | [1, 4NF, 8, 8] | [1, 4NF, 4, 4] | 4NF × 4² = **64·NF** |
| **Decoder** |
| DeConv1 | TransConv+BN+ReLU | [1, 4NF, 4, 4] | [1, 2NF, 8, 8] | 2NF × 8² = **128·NF** |
| DeConv2 | TransConv+BN+ReLU | [1, 2NF, 8, 8] | [1, NF, 16, 16] | NF × 16² = **256·NF** |
| DeConv3 | TransConv+BN+ReLU | [1, NF, 16, 16] | [1, NC, 32, 32] | NC × 32² = **1024·NC** |
| **Classifier** |
| Conv1×1 | Conv2d | [1, NC, 32, 32] | [1, NC, 32, 32] | NC × 32² = **1024·NC** |

### Total Outputs (32×32):
```
O_total = 1024·NF + 256·NF + 512·NF + 128·NF + 256·NF + 64·NF + 128·NF + 256·NF + 1024·NC + 1024·NC

O_total = NF(1024 + 256 + 512 + 128 + 256 + 64 + 128 + 256) + NC(1024 + 1024)

O_total = NF(2624) + NC(2048)

O_total = 2624·NF + 2048·NC
```

#### Example (NF=32, NC=24):
```
O_total = 2624(32) + 2048(24)
        = 83,968 + 49,152
        = 133,120 activations
```

---

## Step 3: Number of Connections

**Connection Definition:** Each weight parameter connects to multiple input activation elements across spatial locations. For a convolutional layer, each filter is applied to every spatial position in the input feature map.

### Formula:
For a Conv2d layer with output spatial size (H_out, W_out):
```
Connections = (# weights) × (# output spatial locations)
Connections = (in_c × out_c × k_h × k_w) × (H_out × W_out)
```

**Reasoning:** Each of the (in_c × out_c × k_h × k_w) weights is used at every output position (H_out × W_out).

### Encoder Layer 1: Conv2d, Output [NF, 32, 32]
```
C₁ = W₁ × (32 × 32)
C₁ = (NIC·NF·k²) × 1024
C₁ = 1024·NIC·NF·k²
```
- Each of the NIC·NF·k² weights connects to all 1024 output positions

### Encoder Layer 2: Conv2d, Output [2NF, 16, 16]
```
C₂ = W₂ × (16 × 16)
C₂ = (2NF²·k²) × 256
C₂ = 512·NF²·k²
```
- Each of the 2NF²·k² weights connects to all 256 output positions

### Encoder Layer 3: Conv2d, Output [4NF, 8, 8]
```
C₃ = W₃ × (8 × 8)
C₃ = (8NF²·k²) × 64
C₃ = 512·NF²·k²
```
- Each of the 8NF²·k² weights connects to all 64 output positions

### Decoder Layer 1: ConvTranspose2d, Output [2NF, 8, 8]
For transpose convolution: each weight operates on input positions
```
C₄ = W₄ × (4 × 4)  [input spatial size]
C₄ = (16NF²) × 16
C₄ = 256·NF²
```
- The 16NF² weights operate on the 4×4 input to produce 8×8 output

### Decoder Layer 2: ConvTranspose2d, Output [NF, 16, 16]
```
C₅ = W₅ × (8 × 8)  [input spatial size]
C₅ = (4NF²) × 64
C₅ = 256·NF²
```

### Decoder Layer 3: ConvTranspose2d, Output [NC, 32, 32]
```
C₆ = W₆ × (16 × 16)  [input spatial size]
C₆ = (4NF·NC) × 256
C₆ = 1024·NF·NC
```

### Classifier: Conv2d(1×1), Output [NC, 32, 32]
```
C₇ = W₇ × (32 × 32)
C₇ = (NC²) × 1024
C₇ = 1024·NC²
```

### Total Connections (32×32):
```
C_total = C₁ + C₂ + C₃ + C₄ + C₅ + C₆ + C₇

C_total = 1024·NIC·NF·k² + 512·NF²·k² + 512·NF²·k² + 256·NF² + 256·NF² + 1024·NF·NC + 1024·NC²

C_total = 1024·NIC·NF·k² + 1024·NF²·k² + 512·NF² + 1024·NF·NC + 1024·NC²

C_total = 1024·NF·k²(NIC + NF) + 512·NF² + 1024·NF·NC + 1024·NC²
```

**Factored form:**
```
C_total = 1024[NF·k²(NIC + NF) + NF·NC + NC²] + 512·NF²
```

#### For k=3:
```
C_total = 9216·NF(NIC + NF) + 512·NF² + 1024·NF·NC + 1024·NC²
```

#### For k=5:
```
C_total = 25600·NF(NIC + NF) + 512·NF² + 1024·NF·NC + 1024·NC²
```

#### Example (NIC=1, NF=32, NC=24, k=3):
```
C_total = 9216(32)(1 + 32) + 512(32²) + 1024(32)(24) + 1024(24²)
        = 9216(32)(33) + 512(1024) + 1024(768) + 1024(576)
        = 9,732,096 + 524,288 + 786,432 + 589,824
        = 11,632,640 connections
```

---

# Part 2: Analysis for H=W=64 (64×64 Input)

## Architecture Flow for 64×64:

| Layer | Input Shape | Output Shape | Spatial Size Change |
|-------|-------------|--------------|---------------------|
| Input | [1, NIC, 64, 64] | - | - |
| Conv1+Pool1 | [1, NIC, 64, 64] | [1, NF, 32, 32] | ÷2 |
| Conv2+Pool2 | [1, NF, 32, 32] | [1, 2NF, 16, 16] | ÷2 |
| Conv3+Pool3 | [1, 2NF, 16, 16] | [1, 4NF, 8, 8] | ÷2 |
| DeConv1 | [1, 4NF, 8, 8] | [1, 2NF, 16, 16] | ×2 |
| DeConv2 | [1, 2NF, 16, 16] | [1, NF, 32, 32] | ×2 |
| DeConv3 | [1, NF, 32, 32] | [1, NC, 64, 64] | ×2 |
| Classifier | [1, NC, 64, 64] | [1, NC, 64, 64] | ×1 |

---

## Step 1: Number of Weights (64×64)

**Key Insight:** The number of weights is **independent of input spatial size** because:
- Convolutional filters are defined by (in_channels, out_channels, kernel_height, kernel_width)
- These dimensions don't change with input size

```
W_total(64×64) = W_total(32×32) = NIC·NF·k² + 10NF²·k² + 20NF² + 4NF·NC + NC²
```

#### For k=3:
```
W_total = 9·NIC·NF + 110NF² + 4NF·NC + NC²
```

#### Example (NIC=1, NF=32, NC=24, k=3):
```
W_total = 116,576 weights (SAME as 32×32)
```

**Scaling Factor:** W₆₄/W₃₂ = **1.0×**

---

## Step 2: Number of Outputs (64×64)

### Spatial Dimension Tracking:

| Layer | Output Shape | # Elements |
|-------|--------------|------------|
| Conv1 | [1, NF, 64, 64] | NF × 64² = **4096·NF** |
| Pool1 | [1, NF, 32, 32] | NF × 32² = **1024·NF** |
| Conv2 | [1, 2NF, 32, 32] | 2NF × 32² = **2048·NF** |
| Pool2 | [1, 2NF, 16, 16] | 2NF × 16² = **512·NF** |
| Conv3 | [1, 4NF, 16, 16] | 4NF × 16² = **1024·NF** |
| Pool3 | [1, 4NF, 8, 8] | 4NF × 8² = **256·NF** |
| DeConv1 | [1, 2NF, 16, 16] | 2NF × 16² = **512·NF** |
| DeConv2 | [1, NF, 32, 32] | NF × 32² = **1024·NF** |
| DeConv3 | [1, NC, 64, 64] | NC × 64² = **4096·NC** |
| Classifier | [1, NC, 64, 64] | NC × 64² = **4096·NC** |

### Total Outputs (64×64):
```
O_total = 4096·NF + 1024·NF + 2048·NF + 512·NF + 1024·NF + 256·NF + 512·NF + 1024·NF + 4096·NC + 4096·NC

O_total = NF(4096 + 1024 + 2048 + 512 + 1024 + 256 + 512 + 1024) + NC(4096 + 4096)

O_total = NF(10496) + NC(8192)

O_total = 10496·NF + 8192·NC
```

#### Example (NF=32, NC=24):
```
O_total = 10496(32) + 8192(24)
        = 335,872 + 196,608
        = 532,480 activations
```

**Scaling Factor:**
```
O₆₄/O₃₂ = (10496·NF + 8192·NC) / (2624·NF + 2048·NC)
        = (10496/2624)·NF/(NF+γ·NC) [where γ = NC/NF ratio]
        ≈ 4.0× (exactly 4× for each term)
```

---

## Step 3: Number of Connections (64×64)

### Connection Calculations:

| Layer | Weights | Output/Input Spatial | Connections |
|-------|---------|---------------------|-------------|
| Conv1 | NIC·NF·k² | 64² = 4096 | **4096·NIC·NF·k²** |
| Conv2 | 2NF²·k² | 32² = 1024 | **2048·NF²·k²** |
| Conv3 | 8NF²·k² | 16² = 256 | **2048·NF²·k²** |
| DeConv1 | 16NF² | 8² = 64 (input) | **1024·NF²** |
| DeConv2 | 4NF² | 16² = 256 (input) | **1024·NF²** |
| DeConv3 | 4NF·NC | 32² = 1024 (input) | **4096·NF·NC** |
| Classifier | NC² | 64² = 4096 | **4096·NC²** |

### Total Connections (64×64):
```
C_total = 4096·NIC·NF·k² + 2048·NF²·k² + 2048·NF²·k² + 1024·NF² + 1024·NF² + 4096·NF·NC + 4096·NC²

C_total = 4096·NIC·NF·k² + 4096·NF²·k² + 2048·NF² + 4096·NF·NC + 4096·NC²

C_total = 4096·NF·k²(NIC + NF) + 2048·NF² + 4096·NF·NC + 4096·NC²
```

**Relation to 32×32:**
```
C_total(64×64) = 4 × [1024·NF·k²(NIC + NF) + 512·NF² + 1024·NF·NC + 1024·NC²]
C_total(64×64) = 4 × C_total(32×32)
```

#### For k=3:
```
C_total = 36864·NF(NIC + NF) + 2048·NF² + 4096·NF·NC + 4096·NC²
```

#### Example (NIC=1, NF=32, NC=24, k=3):
```
C_total = 36864(32)(33) + 2048(1024) + 4096(768) + 4096(576)
        = 38,928,384 + 2,097,152 + 3,145,728 + 2,359,296
        = 46,530,560 connections
```

**Scaling Factor:** C₆₄/C₃₂ = **4.0×** (exactly)

---

# Summary Tables

## Table 1: Formulas (General)

| Quantity | 32×32 Input | 64×64 Input | Scaling |
|----------|-------------|-------------|---------|
| **Weights** | NIC·NF·k² + 10NF²·k² + 20NF² + 4NF·NC + NC² | **Same** | **1.0×** |
| **Outputs** | 2624·NF + 2048·NC | 10496·NF + 8192·NC | **4.0×** |
| **Connections** | 1024·NF·k²(NIC+NF) + 512·NF² + 1024·NF·NC + 1024·NC² | 4096·NF·k²(NIC+NF) + 2048·NF² + 4096·NF·NC + 4096·NC² | **4.0×** |

## Table 2: Numerical Example (NIC=1, NF=32, NC=24, k=3)

| Quantity | 32×32 Input | 64×64 Input | Ratio |
|----------|-------------|-------------|-------|
| **Weights** | 116,576 | 116,576 | 1.00× |
| **Outputs** | 133,120 | 532,480 | 4.00× |
| **Connections** | 11,632,640 | 46,530,560 | 4.00× |

## Table 3: Kernel Size Comparison (32×32, NIC=1, NF=32, NC=24)

| Metric | k=3 | k=5 | Ratio (k=5/k=3) |
|--------|-----|-----|-----------------|
| **Weights** | 116,576 | 280,576 | 2.41× |
| **Outputs** | 133,120 | 133,120 | 1.00× |
| **Connections** | 11,632,640 | 28,888,640 | 2.48× |

---

# Key Insights

1. **Weights are Input-Size Independent:** The number of learnable parameters depends only on the architecture hyperparameters (NIC, NF, NC, k), not the input spatial dimensions.

2. **Outputs Scale Quadratically:** When spatial dimensions double (32→64), the number of activation elements quadruples (scales with H²×W²).

3. **Connections Scale Quadratically:** Connections = Weights × Spatial Locations, so they also scale by 4× when input size doubles.

4. **Kernel Size Impact:** Larger kernels (k=5 vs k=3) significantly increase weights and connections for convolutional layers but don't affect outputs.

5. **Memory vs Computation Trade-off:**
   - Larger inputs (64×64) require 4× more memory for activations
   - Larger inputs require 4× more computation (connections)
   - But model size (weights) remains constant

# WandB Links
https://wandb.ai/gracygarg6516-iiit-hyderabad/cifar10-colorization-sweep?nw=nwusergracygarg6516